In [1]:
import catboost
import numpy as np
import pandas as pd

In [2]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow_tracking.db")

mlflow.set_experiment("CatBoost_Lags")

<Experiment: artifact_location='/home/alxdrzd/alxdrzd/ds-projects/restaurant-visitor-eda/notebooks/mlruns/3', creation_time=1781855763781, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1781855763781, lifecycle_stage='active', name='CatBoost_Lags', tags={}, trace_location=None, workspace='default'>

In [3]:
from restaurant_visitor_eda.config import PROCESSED_DATA_DIR

df_train = pd.read_csv(PROCESSED_DATA_DIR / "train_features.csv", parse_dates=["visit_date"])
df_test = pd.read_csv(PROCESSED_DATA_DIR / "test_features.csv", parse_dates=["visit_date"])

print(f"Train shape: {df_train.shape}")
print(f"Test shape: {df_test.shape}")

2026-06-19 11:34:34.970 | INFO     | restaurant_visitor_eda.config:<module>:11 - PROJ_ROOT path is: /home/alxdrzd/alxdrzd/ds-projects/restaurant-visitor-eda


Train shape: (252108, 28)
Test shape: (32019, 28)


In [ ]:
import mlflow
import pandas as pd
from sklearn.metrics import root_mean_squared_log_error

from restaurant_visitor_eda.features import (
    binary_features,
    categorical_features,
    compute_calendar_lags,
    get_custom_cv_splits,
    numeric_features,
    predict_recursive,
)

numeric_features_lags = numeric_features + ["lag_1", "lag_7", "lag_14"]

features = categorical_features + numeric_features_lags + binary_features

df_train_full = compute_calendar_lags(df_train, lags=[1, 7, 14])

for lag in [1, 7, 14]:
    df_train_full[f"lag_{lag}"] = (
        df_train_full[f"lag_{lag}"]
        .fillna(df_train_full["store_dow_mean_cum"])
        .fillna(df_train_full["store_mean_cum"])
    )

splits = get_custom_cv_splits(df_train_full, n_splits=3, val_days=39)
scores = []

model_params = {
    "iterations": 800,
    "learning_rate": 0.03,
    "depth": 6,
    "l2_leaf_reg": 5,
    "loss_function": "RMSE",
    "eval_metric": "RMSE",
    "random_seed": 42,
}

with mlflow.start_run(run_name="CatBoost_Recursive_CV_Lags"):
    mlflow.log_params(model_params)
    mlflow.log_param("num_features", len(features))

    for fold, (train_idx, val_idx) in enumerate(splits):
        train_fold = df_train_full.iloc[train_idx].copy()
        val_fold = df_train_full.iloc[val_idx].copy()

        X_train = train_fold[features]
        y_train = np.log1p(train_fold["visitors"])

        fold_model = catboost.CatBoostRegressor(cat_features=categorical_features, **model_params)

        fold_model.fit(X_train, y_train, verbose=200)

        df_cv_all = pd.concat([train_fold, val_fold], ignore_index=True)

        val_mask = df_cv_all["visit_date"].isin(val_fold["visit_date"])
        df_cv_all.loc[val_mask, "visitors"] = np.nan

        val_start = val_fold["visit_date"].min()
        val_end = val_fold["visit_date"].max()

        df_cv_pred = predict_recursive(
            model=fold_model,
            df_all=df_cv_all,
            start_date=val_start,
            end_date=val_end,
            features=features,
        )

        y_pred = df_cv_pred.loc[val_mask, "visitors"].values
        y_true = val_fold["visitors"].values

        y_pred = np.clip(y_pred, 0, None)

        score = root_mean_squared_log_error(y_true, y_pred)
        scores.append(score)

        mlflow.log_metric(f"val_rmsle_fold_{fold + 1}", score)
        print(f"Fold {fold + 1} RMSLE (Recursive): {score:.4f}")

    mean_score = np.mean(scores)
    mlflow.log_metric("val_rmsle_mean", mean_score)
    print(f"\nMean RMSLE (Recursive): {mean_score:.4f}")

    X_train_final = df_train_full[features]
    y_train_final = np.log1p(df_train_full["visitors"])

    final_model = catboost.CatBoostRegressor(cat_features=categorical_features, **model_params)

    final_model.fit(X_train_final, y_train_final, verbose=200)

    df_test_copy = df_test.copy()

    for lag in [1, 7, 14]:
        if f"lag_{lag}" not in df_test_copy.columns:
            df_test_copy[f"lag_{lag}"] = np.nan

    df_test_copy["visitors"] = np.nan

    df_all_final = pd.concat([df_train_full, df_test_copy], ignore_index=True)
    df_all_final = df_all_final.sort_values(["air_store_id", "visit_date"]).reset_index(drop=True)

    test_start = df_test_copy["visit_date"].min()
    test_end = df_test_copy["visit_date"].max()

    df_final_pred = predict_recursive(
        model=final_model,
        df_all=df_all_final,
        start_date=test_start,
        end_date=test_end,
        features=features,
    )

    test_mask = (df_final_pred["visit_date"] >= test_start) & (
        df_final_pred["visit_date"] <= test_end
    )
    df_test_predicted = df_final_pred[test_mask]

    df_test_ordered = pd.merge(
        df_test[["air_store_id", "visit_date"]],
        df_test_predicted[["air_store_id", "visit_date", "visitors"]],
        on=["air_store_id", "visit_date"],
        how="left",
    )

    test_preds_clipped = np.clip(df_test_ordered["visitors"], 1.0, None)

    submission = pd.DataFrame(
        {
            "id": df_test["air_store_id"] + "_" + df_test["visit_date"].dt.strftime("%Y-%m-%d"),
            "visitors": test_preds_clipped,
        }
    )

    print("Minimal predicted value:", submission["visitors"].min())

    submission_path = "submission_catboost_lags.csv"
    submission.to_csv(submission_path, index=False)

    mlflow.catboost.log_model(final_model, "final_catboost_model")
    mlflow.log_artifact(submission_path)

submission.head()

Fold 1: Train ends 2017-03-14| Val: 2017-03-15 to 2017-04-22
Fold 2: Train ends 2017-02-03| Val: 2017-02-04 to 2017-03-14
Fold 3: Train ends 2016-12-26| Val: 2016-12-27 to 2017-02-03
0:	learn: 0.7903833	total: 111ms	remaining: 1m 28s
200:	learn: 0.5172869	total: 14.3s	remaining: 42.7s
400:	learn: 0.5050637	total: 54.1s	remaining: 53.8s
600:	learn: 0.4969005	total: 1m 53s	remaining: 37.5s
799:	learn: 0.4906931	total: 1m 59s	remaining: 0us
Fold 1 RMSLE (Recursive): 0.5798
0:	learn: 0.7920940	total: 35.8ms	remaining: 28.6s
200:	learn: 0.5189880	total: 10.7s	remaining: 31.8s
400:	learn: 0.5078131	total: 26.9s	remaining: 26.8s
600:	learn: 0.4992593	total: 44.5s	remaining: 14.7s
799:	learn: 0.4943105	total: 1m 58s	remaining: 0us
Fold 2 RMSLE (Recursive): 0.5125
0:	learn: 0.7925768	total: 39.9ms	remaining: 31.9s
200:	learn: 0.5158016	total: 7.18s	remaining: 21.4s
400:	learn: 0.5049707	total: 20s	remaining: 19.9s
600:	learn: 0.4952230	total: 37.7s	remaining: 12.5s
799:	learn: 0.4907712	total: 

2026/06/19 11:42:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/19 11:43:24 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


,id,visitors
0,air_00a91d42b08b08d9_2017-04-23,5.292441
1,air_08cb3c4ee6cd6a22_2017-04-23,13.281558
2,air_f8233ad00755c35c_2017-04-23,3.625909
3,air_234d3dbf7f3d5a50_2017-04-23,7.696673
4,air_a563896da3777078_2017-04-23,27.117194
